In [3]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
NOISE_STD_FRACTION = 0.05  # 5% of historical std deviation
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order, excluding targets
original_order = [col for col in df.columns if col not in TARGET_COLUMNS]

# Determine dynamic features (forecasted ones)
excluded_cols = set(STATIC_FEATURES + TARGET_COLUMNS + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        model = Prophet(yearly_seasonality=True, daily_seasonality=False, weekly_seasonality=False)
        model.fit(ts)

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Add Gaussian noise: std_dev * NOISE_STD_FRACTION
        std_dev = ts['y'].std()
        noise = np.random.normal(0, NOISE_STD_FRACTION * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder columns to match original file (excluding targets)
    final_columns = [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[['date'] + final_columns]

    # Append to full dataset
    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("synthetic_features_with_noise_2026_2050.csv", index=False)
print("✅ Synthetic dataset with noise saved as 'synthetic_features_with_noise_2026_2050.csv'")


Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]23:38:34 - cmdstanpy - INFO - Chain [1] start processing
23:38:34 - cmdstanpy - INFO - Chain [1] done processing
23:38:34 - cmdstanpy - INFO - Chain [1] start processing
23:38:34 - cmdstanpy - INFO - Chain [1] done processing
23:38:34 - cmdstanpy - INFO - Chain [1] start processing
23:38:34 - cmdstanpy - INFO - Chain [1] done processing
23:38:35 - cmdstanpy - INFO - Chain [1] start processing
23:38:35 - cmdstanpy - INFO - Chain [1] done processing
23:38:35 - cmdstanpy - INFO - Chain [1] start processing
23:38:35 - cmdstanpy - INFO - Chain [1] done processing
23:38:35 - cmdstanpy - INFO - Chain [1] start processing
23:38:35 - cmdstanpy - INFO - Chain [1] done processing
23:38:35 - cmdstanpy - INFO - Chain [1] start processing
23:38:35 - cmdstanpy - INFO - Chain [1] done processing
23:38:35 - cmdstanpy - INFO - Chain [1] start processing
23:38:36 - cmdstanpy - INFO - Chain [1] done processing
23:38:36 - cmdstanpy - INFO - Chain [1] 

✅ Synthetic dataset with noise saved as 'synthetic_features_with_noise_2026_2050.csv'
